In [1]:
import pandas as pd
import numpy as np

In [2]:
shapes = pd.read_csv('./output/future/optimalTODlocation/TOD1_4/shapes.txt')
shapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,10th-ASTH-CSLA,-33.934464,150.740443,1,0.000000
1,10th-ASTH-CSLA,-33.934563,150.741161,2,80.402344
2,10th-ASTH-CSLA,-33.933217,150.741428,3,214.341169
3,10th-ASTH-CSLA,-33.931870,150.741695,4,348.280199
4,10th-ASTH-CSLA,-33.930564,150.741953,5,478.217869


In [3]:
len(shapes['shape_id'].unique())

66

In [4]:
routesdf = shapes[['shape_id','shape_dist_traveled']].groupby('shape_id').max().reset_index()
routesdf.head()

,shape_id,shape_dist_traveled
0,10th-ASTH-CSLA,21765.162282
1,10th-CSLA-ASTH,21589.298425
2,15th-AERO-LVPL,22672.658563
3,15th-LVPL-AERO,22721.130063
4,20th-AERO-LVPL,24913.717360


In [5]:
stopsByRoute = pd.read_csv('StopsByRoute_TwaysAndLocalRoutes_TOD14.csv')
stopsByRoute.head()

,shape_id,begin,end,stop_id,stop_name,stop_lat,stop_lon,distAlongR,parent_route
0,10th-ASTH-CSLA,1,427,20200814260,10thAv_AerotropolisSouth_b,-33.934483,150.740965,47.94503,10th-ASTH-CSLA
1,10th-ASTH-CSLA,1,427,20200814198,Rossmore-SW-WE,-33.925766,150.769163,3804.08919,10th-ASTH-CSLA
2,10th-ASTH-CSLA,1,427,20200814202,Rossmore-SE-WE,-33.929035,150.784891,5446.92407,10th-ASTH-CSLA
3,10th-ASTH-CSLA,1,427,20200814210,Austral-SW-WE,-33.931200,150.798671,6743.77943,10th-ASTH-CSLA
4,10th-ASTH-CSLA,1,427,20200814214,Austral-SE-WE,-33.933048,150.811829,7977.77206,10th-ASTH-CSLA


In [6]:
stopsByRoute = stopsByRoute[['parent_route','stop_id']].groupby('parent_route',as_index=False).count()
stopsByRoute.head()

,parent_route,stop_id
0,10th-ASTH-CSLA,13
1,10th-CSLA-ASTH,13
2,15th-AERO-LVPL,13
3,15th-LVPL-AERO,13
4,20th-AERO-LVPL,14


In [7]:
routesdf = routesdf.merge(stopsByRoute, how='left', left_on='shape_id', right_on='parent_route').drop('parent_route', axis=1)
routesdf.columns = ['shape_id','shape_length','stops']
routesdf.head()

,shape_id,shape_length,stops
0,10th-ASTH-CSLA,21765.162282,13
1,10th-CSLA-ASTH,21589.298425,13
2,15th-AERO-LVPL,22672.658563,13
3,15th-LVPL-AERO,22721.130063,13
4,20th-AERO-LVPL,24913.717360,14


In [8]:
# From combined travel time model (Liverpool bus routes)

routesdf['traveltime_min'] = np.round(((0.0262*routesdf['shape_length']/1000) + (0.0055*routesdf['stops']))*60,0)
routesdf.head()

,shape_id,shape_length,stops,traveltime_min
0,10th-ASTH-CSLA,21765.162282,13,39.0
1,10th-CSLA-ASTH,21589.298425,13,38.0
2,15th-AERO-LVPL,22672.658563,13,40.0
3,15th-LVPL-AERO,22721.130063,13,40.0
4,20th-AERO-LVPL,24913.717360,14,44.0


In [9]:
routesdf.isnull().sum()

shape_id          0
shape_length      0
stops             0
traveltime_min    0
dtype: int64

In [10]:
routesdf.to_csv('traveltimes_TwaysAndLocalRoutes_TOD14.csv', index=False)